# Streaming Lakehouse with ArcFlow — Thinking Like a Software Engineer

> **Module 2 · Part 2** | ~30 min | Level 300

In Part 1 you built a raw → bronze → silver pipeline with plain PySpark. It works — but is it production-ready?

In this notebook you will take the **exact same transforms** and run them through **ArcFlow**, an open-source PySpark streaming ELT framework. The goal is not to learn a specific tool, but to see how **packaging your code as software** changes everything:

| Principle | What it means for you |
|-----------|----------------------|
| 🧪 **Testability** | Validate transforms without writing to Delta |
| 📊 **Observability** | Health of every stream in one call |
| 🔌 **Extensibility** | Add new tables by writing only business logic |
| 🔍 **Debuggability** | Automatic [Spark job descriptions](https://spark.apache.org/docs/latest/web-ui.html) in the Spark UI |

**Prerequisites** — Complete Part 1 (or at least be familiar with the transforms). The `stream_bronze_and_silver` Spark Job Definition should already be running.

---

## 1 — The Problem with Static Notebook Code

Think back to the bronze transform you built in Part 1. It worked perfectly for **one table**. Now imagine adding the rest:

| Entity | Format | Envelope | Custom Logic |
|--------|--------|----------|-------------|
| shipment_scan_event | [Eventstream](https://learn.microsoft.com/fabric/real-time-intelligence/event-streams/overview) (Kafka) | `_meta` + `data[]` | Explode + flatten 30+ fields |
| shipment | JSON files | `_meta` + `data[]` | Flatten nested addresses |
| order | JSON files | `_meta` + `data[]` | Explode `OrderLines[]` to line-level grain |
| item | JSON files | `_meta` + `data[]` | Cast types, validate SKU |
| customer | Parquet files | Flat | Flatten `DeliveryAddress` / `BillingAddress` structs |
| route | Parquet files | Flat | Dimensional lookup table |
| facility | Parquet files | Flat | Dimensional lookup table |
| servicelevel | Parquet files | Flat | Dimensional lookup table |
| exceptiontype | Parquet files | Flat | Dimensional lookup table |

With static notebook code, each table needs its own:
- Schema definition
- Reader configuration
- Transform function
- `writeStream` boilerplate (checkpoint, trigger, output table)
- Monitoring code

**What breaks at scale:**

| Pain point | Why it matters |
|-----------|----------------|
| ❌ Duplicated boilerplate | Copy-paste errors compound across tables |
| ❌ No testability | You must write to Delta to see if your transform works |
| ❌ No versioning | The transform *is* the notebook — impossible to package or distribute |
| ❌ Schema changes cascade | One upstream change means editing many cells |
| ❌ No observability | Checking stream health means running separate cells per table |

> _Slides supplement this section with a deeper look at framework design principles._

---

## 2 — Packaging Your ELT Framework

**ArcFlow** is a PySpark streaming ELT framework that separates *what* to process (configuration) from *how* to process (framework). Here is the conceptual split:

| You define (metadata) | Framework handles (machinery) |
|----------------------|------------------------------|
| Table name, schema | Reader selection (files, Kafka, EventHub) |
| Source path or connection | [Checkpoint management](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#recovering-from-failures-with-checkpointing) |
| Zone config (append vs. upsert) | `writeStream` orchestration |
| Custom transform function name | [`foreachBatch`](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#foreachbatch) wiring |
| Trigger mode | Stream lifecycle, error handling |

### ⚖️ Side-by-Side: Part 1 vs. ArcFlow

**Part 1 — Raw Spark:**
```python
sse_schema = spark.read.json('Files/archive/shipment_scan_event', ...).schema

def write_bronze_sse(batch_df, batch_id):
    transformed = bronze_transform(batch_df)
    transformed.write.format('delta').mode('append').saveAsTable('explore.bronze_shipment_scan_event')

sse_stream = spark.readStream.schema(sse_schema).json('Files/archive/shipment_scan_event', ...)
query = (sse_stream.writeStream
    .foreachBatch(write_bronze_sse)
    .option('checkpointLocation', 'Files/explore/checkpoints/bronze_sse')
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()
```

**ArcFlow — Config-driven:**
```python
# Configuration (FlowConfig dataclass)
tables['shipment_scan_event'] = FlowConfig(
    name='shipment_scan_event',
    format='json',
    source_uri='Files/landing/shipment_scan_event',
    schema=sse_schema,
    zones={
        'bronze': StageConfig(mode='append', custom_transform='explode_data'),
        'silver': StageConfig(mode='append', custom_transform='silver_shipment_scan_event'),
    }
)

# Execution — one line runs all tables through all zones
controller.run_zone_pipeline('bronze')
```

> The business logic (transforms) is identical. The boilerplate (readers, writers, checkpoints, triggers) disappears into the framework.

> _Slides supplement this section with an introduction to ArcFlow's architecture._

---

## 3 — Add a New Object to the Framework

This is where the payoff becomes real. We will:

1. Register the same transforms from Part 1 as **named transformer functions**
2. Use ArcFlow's **test methods** (memory sink) to validate without writing to Delta
3. Run the pipeline through the **controller** with automatic Spark job descriptions
4. Observe all streams at once with `controller.get_status()`
5. Wire up **new tables** by writing only business logic

### 3.1 Register a Transformer Function

In Part 1, transforms were loose functions in your notebook. In ArcFlow, you register them with the [`@register_zone_transformer`](https://github.com/microsoft/arcflow) decorator so the framework can look them up from configuration.

Execute the cell below to register two transformers:

In [ ]:
from arcflow.transformations import register_zone_transformer
from pyspark.sql.functions import explode, col
import pyspark.sql.functions as sf

@register_zone_transformer
def explode_data(df):
    """Bronze transform: explode the _meta + data[] envelope into individual records."""
    return df.select(
        col('_meta.*'),
        explode('data').alias('data'),
        col('_processing_timestamp')
    )

@register_zone_transformer
def silver_shipment_scan_event(df):
    """Silver transform: flatten the shipment scan event record."""
    return df.selectExpr('_meta.*', 'data.*').drop('_meta', 'data', '_processing_timestamp')

print('Transformers registered.')

The `@register_zone_transformer` decorator stores the function in a global registry. When ArcFlow sees `custom_transform='explode_data'` in a `StageConfig`, it looks up this function by name.

> The code inside the transformer is the **same business logic** from Part 1. The decorator is the only addition.

### 3.2 🧪 Test Without Writing to Delta

Remember the [`memory` sink](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#output-sinks) pattern from Part 1? ArcFlow wraps it into reusable test methods on `ZonePipeline`.

Execute the cells below to see three levels of inspection — all using the memory sink:

In [ ]:
from arcflow.pipelines.zone_pipeline import ZonePipeline
from arcflow.config import get_config
from pipeline_config import get_tables  # your project's table registry

config = get_config({
    'landing_uri': 'Files/landing',
    'archive_uri': 'Files/archive',
    'checkpoint_uri': 'Files/checkpoints',
})

tables = get_tables(config)
pipeline = ZonePipeline(spark, zone='bronze', config=config)

# test_input: raw payload + all connector metadata (no transforms applied)
df_raw = pipeline.test_input(tables['shipment_scan_event'], raw=True)
display(df_raw)

In [ ]:
# test_input: with schema parsing (from_json applied, but no custom transform)
df_parsed = pipeline.test_input(tables['shipment_scan_event'])
display(df_parsed)

In [ ]:
# test_output: full transform chain — schema + custom_transform + normalization + audit
df_transformed = pipeline.test_output(tables['shipment_scan_event'])
display(df_transformed)

Three levels of inspection, all using the `memory` sink — no Delta writes, no checkpoints, no cleanup. The same pattern works for **any** source type (files, Kafka, EventHub).

| Method | What you see |
|--------|--------------|
| `test_input(raw=True)` | Raw payload + connector metadata (no transforms) |
| `test_input()` | Schema parsed, but no custom transform |
| `test_output()` | Full transform chain — schema + custom + normalization + audit |

Compare this to Part 1 where you had to manually build each step.

### 3.3 Run the Pipeline via the Controller

The controller orchestrates all tables through their configured zones. Execute the cell below:

In [ ]:
from arcflow.controller import Controller

controller = Controller(spark, config, tables)

# Run all tables through the bronze zone
controller.run_zone_pipeline('bronze')
controller.await_completion()

Behind the scenes, the controller:
- Selected the correct reader for each table's format (JSON files, Parquet files, Kafka)
- Applied each table's `custom_transform` via [`foreachBatch`](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#foreachbatch)
- Set **[Spark job descriptions](https://spark.apache.org/docs/latest/web-ui.html#jobs-tab)** automatically so you can trace each stream in the Spark UI

```python
spark.sparkContext.setJobDescription(f'Running OPTIMIZE on {table.name}')
```

> Framework design principle: **visibility and debuggability should be built in, not bolted on.**

### 3.4 📊 Observability Across All Streams

In Part 1, you checked `query.status` one stream at a time. The controller gives you a single view.

Execute the cell below:

In [ ]:
# Health of every running or completed StreamingQuery in one call
controller.get_status()

> *Learning moment: the code is nearly identical — the packaging is what changes everything.*
>
> Easier to test, version, distribute, and hand to a teammate.

### 3.5 🎯 Guided Exercise: Wire Up a New Table

Let's add `order` and `customer` to the framework. You write **only the business logic** — the transformer function. The framework handles everything else.

**Your task:**

1. Register a `silver_order` transformer that explodes the `data[]` array and then explodes the nested `OrderLines` array to produce one row per order line
2. Register a `silver_customer` transformer that flattens the `DeliveryAddress` and `BillingAddress` nested structs

**💡 Hints:**
- Order has `_meta` + `data[]` envelope (JSON). Each record in `data` has an `OrderLines` array.
- Customer is Parquet (flat) — no `_meta`/`data[]` envelope, but `DeliveryAddress` and `BillingAddress` are nested structs.

Fill in the transformer bodies in the cell below:

In [ ]:
# Your transformer functions here

@register_zone_transformer
def silver_order(df):
    """Silver transform for orders: explode to order-line grain."""
    # TODO: implement
    pass

@register_zone_transformer
def silver_customer(df):
    """Silver transform for customers: flatten address structs."""
    # TODO: implement
    pass

---

<details>
  <summary><strong>🔑 Solution:</strong> Click to reveal the answer</summary>

<br/>

**Approach:** Use the familiar explode → expand pattern from Part 1, but now as registered transformers:

```python
@register_zone_transformer
def silver_order(df):
    """Silver transform for orders: explode to order-line grain."""
    # Step 1: explode the data[] envelope
    exploded = df.select(col('_meta.*'), explode('data').alias('order'), col('_processing_timestamp'))
    # Step 2: explode OrderLines within each order
    with_lines = exploded.select(
        col('order.OrderId'),
        col('order.OrderNumber'),
        col('order.OrderDate'),
        col('order.OrderTotal'),
        col('order.Source'),
        col('order.CustomerId'),
        explode(col('order.OrderLines')).alias('line'),
        col('_processing_timestamp')
    )
    # Step 3: flatten line struct
    return with_lines.select('OrderId', 'OrderNumber', 'OrderDate', 'OrderTotal',
                             'Source', 'CustomerId', 'line.*', '_processing_timestamp')

@register_zone_transformer
def silver_customer(df):
    """Silver transform for customers: flatten address structs."""
    return (df
        .select(
            '*',
            col('DeliveryAddress.*'),
            col('BillingAddress.*'),
        )
        .drop('DeliveryAddress', 'BillingAddress')
    )
```

**Key Takeaway:** The transform code is the same complexity as Part 1 — but now it's named, registered, testable, and reusable across environments.

</details>

**Validate your transforms with the test methods** — execute the cells below:

In [ ]:
# Test the order silver transform
silver_pipeline = ZonePipeline(spark, zone='silver', config=config)
df_order = silver_pipeline.test_output(tables['order'])
display(df_order)

In [ ]:
# Test the customer silver transform
df_customer = silver_pipeline.test_output(tables['customer'])
display(df_customer)

Now apply the changes to the running Spark Job Definition:

1. Update the `FlowConfig` for `order` and `customer` to include a `silver` zone with your new transformer names
2. Validate in this notebook with `controller.run_zone_pipeline('silver')`
3. Stop and restart the Spark Job Definition to apply the new configuration in production

> *Learning moment: the framework absorbs complexity; you're writing software, not scripts.*

---

## 4 — Notebooks vs. Spark Job Definitions

You've been developing in a **Notebook**. The production pipeline runs as a [**Spark Job Definition (SJD)**](https://learn.microsoft.com/fabric/data-engineering/create-spark-job-definition). When should you graduate?

| | Notebook | Spark Job Definition |
|---|---------|---------------------|
| **Purpose** | Development, exploration, debugging | Production execution |
| **Interactivity** | Cell-by-cell, visual output | Headless, scheduled or triggered |
| **State** | Session-scoped | Runs to completion, restartable |
| **Configuration** | Inline, ad-hoc | Source-controlled, versioned |
| **Monitoring** | Manual cell execution | [Spark UI](https://spark.apache.org/docs/latest/web-ui.html), `get_status()` |
| **Scaling** | Single session | Long-running, auto-restart capable |

**Rule of thumb:**
- **Notebook** for building, testing, and one-off runs
- **SJD** for anything that runs on a schedule or continuously

The transition is simple when your code is packaged: the SJD imports the same framework and config — no rewriting.

---

## 5 — Dimensional Modeling

The silver layer gives you clean, conformed data. The **gold layer** shapes it for analytics — [star schemas](https://learn.microsoft.com/fabric/data-warehouse/dimensional-modeling-overview), slowly changing dimensions (SCD2), and fact tables.

### The Framework Advantage

SCD2 logic is complex — tracking history, managing surrogate keys, handling effective dates. Writing it once is hard enough. Writing it correctly for every dimension table is a maintenance burden.

**Without a framework:**
```python
# Copy-paste this 50-line MERGE statement for every dimension table
# Hope nobody introduces a subtle bug in copy #7
# Good luck upgrading the SCD2 logic across all copies
```

**With a framework:**
```python
# You define WHAT makes this dimension unique
DimensionConfig(
    name='dim_customer',
    source_zone='silver',
    target_zone='gold',
    business_keys=['customer_id'],
    transform='dim_customer_transform',
)

# The framework handles HOW to build and maintain the SCD2 table
controller.run_dimensions()
```

**The principle:** separate *what* an object is (your config) from *how* it is built (the framework). Don't repeat SCD2 logic — it's hard to maintain, hard to upgrade, and hard to audit.

> _Slides supplement this section with SCD2 implementation details and dimensional modeling patterns._

---

## 🏆 Key Takeaways

| Concept | What you practiced |
|---------|-----------|
| **Static code → packaged framework** | Same transforms, but testable, versionable, distributable |
| **`@register_zone_transformer`** | Name your transforms so config can reference them |
| **`test_input` / `test_output`** | Memory sink pattern wrapped as a reusable method |
| **`controller.run_zone_pipeline()`** | One call runs all tables through a zone |
| **`controller.get_status()`** | Observe every stream's health in one call |
| **Spark job descriptions** | Built-in debuggability in the Spark UI |
| **Notebook → SJD** | Develop interactively, deploy as a scheduled job |
| **Dimensional modeling** | Separate SCD2 machinery from business logic |

### 📚 Further Reading

- [Spark Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html)
- [Fabric Spark Job Definitions](https://learn.microsoft.com/fabric/data-engineering/create-spark-job-definition)
- [Delta Lake Streaming](https://docs.delta.io/latest/delta-streaming.html)
- [Fabric Dimensional Modeling](https://learn.microsoft.com/fabric/data-warehouse/dimensional-modeling-overview)

### The Core Lesson

The transforms you wrote in Part 1 and Part 2 are **nearly identical**. The difference is how they are organized:

- Part 1: transforms live in notebook cells, wired together with manual boilerplate
- Part 2: transforms are registered functions, wired together by a framework driven by configuration

When you package your ELT logic as software, you can **test it without side effects**, **version it with git**, **distribute it as a library**, and **hand it to a teammate** who only needs to write business logic.

> ArcFlow is the reference implementation — but the lessons transfer to any framework you build or adopt.